In [37]:
import os
from dotenv import load_dotenv

In [38]:
## Read langsmith key from .env file and setup tracing
loader = load_dotenv()

# LangSmith Tracing Configuration
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "RAG QA Chatbot")

## Read Groq API key from .env file and set it as an environment variable
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
## read Hugging Face token from .env file and set it as an environment variable
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")



In [39]:
## Hugging face Embeddings configuration
# %pip install -q langchain_huggingface
# %pip install -q sentence-transformers
from langchain_huggingface import HuggingFaceEmbeddings
embeddings_model = "all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embeddings_model)



In [40]:
print(embeddings)
print(embeddings.embed_query("Hello world"))

client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
) model_name='all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False show_progress=False
[-0.034477248787879944, 0.031023213639855385, 0.006734968163073063, 0.02610895223915577, -0.03936203941702843, -0.16030244529247284, 0.06692398339509964, -0.006441461853682995, -0.0474504716694355, 0.014758830890059471, 0.07087528705596924, 0.05552760139107704, 0.019193364307284355, -0.026251312345266342, -0.010109573602676392, -0.026940463110804558, 0.022307418286800385, -0.02222663164138794, -0.149692565202713, -0.017493005841970444, 0.007676235865801573, 0.05435224995017052, 0.003254480427131057, 0

In [41]:
## use recursive character text splitter to split the documents into smaller chunks

## create a Chroma vectorstore and add the documents to it
# %pip install -q langchain_chroma
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
vectorstore = Chroma(
        collection_name="pdf_docs",
        embedding_function=embeddings
    )
def store_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    documents = text_splitter.split_documents(documents)
    print(f"Number of documents after splitting: {len(documents)}")
    vectorstore.add_documents(documents)


#create a retriever from the vectorstore
retriever = vectorstore.as_retriever()

#test the retriever with a query
query = "Rajesh's DOB"
retriever.invoke(query)


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


[Document(metadata={'page': 0, 'source': 'data\\Rajesh Biodata.pdf'}, page_content='RAJESH PANDIT \nDate of Birth:  09/10/1999 \nQualification:   B.Tech in Electronic & Communication Engineering \nJob:   Senior Software Engineer at HashedIn by Deloitte \nExperience:     4 years (2.5 years at Persistent System & 1.5 years at Hashedin by Deloitte) \nHeight:   5 Ft 8 Inch \nWeight:  67 Kg \nHobbies:  Cricket \nFAMILY BACKGROUND \nFather’s Name: Shankarjee Pandit  \nMother’s Name:  Urmila Devi Pandit \nCaste:         Kumbhakar / Kumhar (Prajapati) \nSisters: \n❖ Babita Pandit \n❖ Sunita Pandit'),
 Document(metadata={'page': 0, 'source': 'data\\Rajesh Biodata.pdf'}, page_content='RAJESH PANDIT \nDate of Birth:  09/10/1999 \nQualification:   B.Tech in Electronic & Communication Engineering \nJob:   Senior Software Engineer at HashedIn by Deloitte \nExperience:     4 years (2.5 years at Persistent System & 1.5 years at Hashedin by Deloitte) \nHeight:   5 Ft 8 Inch \nWeight:  67 Kg \nHobbies: 

In [42]:
# read from web 
from langchain.schema import Document

import requests
from bs4 import BeautifulSoup

def load_web_page(url):
    response = requests.get(url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        text = soup.get_text()
        return Document(
            page_content="your text here",
            metadata={"source": url}
        )
    else:
        print(f"Failed to load page: {response.status_code}")
        return None
    

In [43]:
## pdf reader using PyMuPDF
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader("./data", glob="**/*.pdf")
documents = loader.load()
print(f"Loaded {len(documents)} documents.")
store_documents(documents)

Loaded 5 documents.
Number of documents after splitting: 26


In [44]:
## GROQ chat model configuration
from langchain_groq import ChatGroq
chat_model = ChatGroq(model="llama-3.1-8b-instant", temperature=0.5)
chat_model.invoke("Hii")
# chat_model

AIMessage(content='How are you today? Is there something I can help you with or would you like to chat?', response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 37, 'total_tokens': 58, 'completion_time': 0.030216762, 'completion_tokens_details': None, 'prompt_time': 0.001695065, 'prompt_tokens_details': None, 'queue_time': 0.051231525, 'total_time': 0.031911827}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-4c3594df-d857-4db1-9d30-04cb63fd708f-0', usage_metadata={'input_tokens': 37, 'output_tokens': 21, 'total_tokens': 58})

In [45]:
## add context prompt
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
context_prompt = """You are a helpful assistant that answers questions based
 on the following retrieved context from a set of PDF documents: <context>
 {context}
 </context>

<user_question>
    {input}
</user_question>
 """

# if you use variable name as input variable name in the prompt template, you you no need to create runnable pass through for it.
context_prompt = ChatPromptTemplate.from_template(context_prompt)

In [46]:
## create a contextual retriever that uses the retriever and the context prompt

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_core.runnables import RunnablePassthrough

rag_document_chain = create_stuff_documents_chain(chat_model, context_prompt)
rag_qa_chain  = create_retrieval_chain(retriever,rag_document_chain)
# 🔥 Mapping layer (important)
# rag_qa_chain = (
#     {
#         "input": RunnablePassthrough(),
#         "question": RunnablePassthrough()
#     }
#     | retrieval_chain
# )
#sample if you use input in prompt
# response = retrieval_chain.invoke({
#     "input": "What is Rajesh's DOB?"
# })

# print(response["answer"])

# retrieval_chain.invoke({"input": "What is Rajesh's DOB?"})
rag_qa_chain.invoke({ "input": "What is Rajesh's DOB?" })



{'input': "What is Rajesh's DOB?",
 'context': [Document(metadata={'page': 0, 'source': 'data\\Rajesh Biodata.pdf'}, page_content='RAJESH PANDIT \nDate of Birth:  09/10/1999 \nQualification:   B.Tech in Electronic & Communication Engineering \nJob:   Senior Software Engineer at HashedIn by Deloitte \nExperience:     4 years (2.5 years at Persistent System & 1.5 years at Hashedin by Deloitte) \nHeight:   5 Ft 8 Inch \nWeight:  67 Kg \nHobbies:  Cricket \nFAMILY BACKGROUND \nFather’s Name: Shankarjee Pandit  \nMother’s Name:  Urmila Devi Pandit \nCaste:         Kumbhakar / Kumhar (Prajapati) \nSisters: \n❖ Babita Pandit \n❖ Sunita Pandit'),
  Document(metadata={'page': 0, 'source': 'data\\Rajesh Biodata.pdf'}, page_content='RAJESH PANDIT \nDate of Birth:  09/10/1999 \nQualification:   B.Tech in Electronic & Communication Engineering \nJob:   Senior Software Engineer at HashedIn by Deloitte \nExperience:     4 years (2.5 years at Persistent System & 1.5 years at Hashedin by Deloitte) \nHe

In [47]:
only_answer_chain = rag_qa_chain | (lambda x: x["answer"])
only_answer_chain.invoke({"input": "What is Rajesh's DOB & height?"})
# only_answer_chain.invoke({"input": "Rajesh Height details  "})

"Rajesh's Date of Birth (DOB) is 09/10/1999 and his height is 5 Ft 8 Inch."

In [48]:
# read from a web page and add it to the vectorstore
web_page_documents = load_web_page("https://wbxpress.com/schedule-general-election-west-bengal-legislative-assembly-2026/")
store_documents([web_page_documents])

Number of documents after splitting: 1


In [49]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains import create_history_aware_retriever
from langchain_core.messages import HumanMessage, SystemMessage
# ✅ MUST be a string
contextualized_system_prompt = """Given a chat history and a follow up question,
use the chat history as context to understand the question.
Formulate a standalone question that can be understood without the chat history.
Do NOT answer the question, just reformulate it if needed.
Otherwise return it as it is.
strictly keep the name and other important details of user
"""

# ✅ Correct prompt
contextualized_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualized_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
    # SystemMessage(content=contextualized_system_prompt),
    # MessagesPlaceholder(variable_name="chat_history"),
    # HumanMessage(content="{input}")
])

# ✅ Chain
history_aware_retrieval_chain = create_history_aware_retriever(
    chat_model,
    retriever,
    contextualized_prompt
)

In [50]:
chatbot_system_prompt = """You are a helpful assistant that answers questions 
based on the retrieved context <context> {context} </context>
Keep the answer short and concise. 
If context or previous chat history does not contain relevant information, say you don't know 
but don't mention that you are using context or chat history to user.
"""

In [51]:
chatboat_prompt = ChatPromptTemplate.from_messages([
    ("system",chatbot_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])
chatbot_question_answer_chain = create_stuff_documents_chain(chat_model, chatboat_prompt)

chatbot_rag_chain = create_retrieval_chain(history_aware_retrieval_chain, chatbot_question_answer_chain)

In [52]:
## History and session implementation
store = {}
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
def get_session_history(session_id: str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


chatbot_conversational_rag_chain = RunnableWithMessageHistory(
    chatbot_rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"
)
    


In [53]:
import uuid

rajesh_session_id = str(uuid.uuid4())
krishna_session_id = str(uuid.uuid4())

class Chatbot:
    def __init__(self, session_id):
        print("Chatbot initialized")
        self.chatbot = chatbot_conversational_rag_chain
        self.session_id = session_id

    def chat(self, message):   # ✅ added self
        response = self.chatbot.invoke(
            {"input": message},
            config={
                "configurable": {"session_id": self.session_id}
            }
        )
        print("You: "+ message)
        print("bot: " + response['answer'])
        return response

In [54]:
##Rajesh Chat
chatbot = Chatbot(rajesh_session_id)

chatbot.chat("Hi")
chatbot.chat("Do you know my name?")
chatbot.chat("I'm that Rajesh Pandit")
chatbot.chat("do you know my year of experience?")
chatbot.chat("do you know krishna marrital status?")

#krishna chat bot
chatbot_kk = Chatbot(krishna_session_id)
chatbot_kk.chat("Hi This is Krishna Pandit")
chatbot_kk.chat("do you know my year of experience?")
chatbot_kk.chat("I'm married now")
chatbot_kk.chat("Siwan is my native village")
chatbot_kk.chat("do you know my marrital status?")

Chatbot initialized
You: Hi
bot: Hello. How can I assist you today?
You: Do you know my name?
bot: I'm not aware of your name.
You: I'm that Rajesh Pandit
bot: Nice to meet you, Rajesh Pandit. I have some information about you from your profile. Would you like to know something specific about yourself?
You: do you know my year of experience?
bot: You have 4 years of experience, with 2.5 years at Persistent System and 1.5 years at HashedIn by Deloitte.
You: do you know krishna marrital status?
bot: I don't know Krishna's marital status.
Chatbot initialized
You: Hi This is Krishna Pandit
bot: I see you're the brother of Rajesh Pandit. How can I assist you today?
You: do you know my year of experience?
bot: I don't know.
You: I'm married now
bot: Congratulations on your marriage. Do you have any children?
You: Siwan is my native village
bot: Your native village is indeed Rampur in Siwan, Bihar.
You: do you know my marrital status?
bot: You mentioned you're married, so your marital status 

{'input': 'do you know my marrital status?',
 'chat_history': [HumanMessage(content='Hi This is Krishna Pandit'),
  AIMessage(content="I see you're the brother of Rajesh Pandit. How can I assist you today?"),
  HumanMessage(content='do you know my year of experience?'),
  AIMessage(content="I don't know."),
  HumanMessage(content="I'm married now"),
  AIMessage(content='Congratulations on your marriage. Do you have any children?'),
  HumanMessage(content='Siwan is my native village'),
  AIMessage(content='Your native village is indeed Rampur in Siwan, Bihar.')],
 'context': [Document(metadata={'source': 'https://wbxpress.com/schedule-general-election-west-bengal-legislative-assembly-2026/'}, page_content='your text here'),
  Document(metadata={'source': 'https://wbxpress.com/schedule-general-election-west-bengal-legislative-assembly-2026/'}, page_content='your text here'),
  Document(metadata={'source': 'https://wbxpress.com/schedule-general-election-west-bengal-legislative-assembly-20